# Feature engineering - data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
# from tabpfn import TabPFNClassifier

C:\Users\Sebastian\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

## How to get df_train and df_test
df_train is used to get x_train and y_train\
df_test is used to get x_test and y_test (for season we want to predict)\
First 7 columns in df_... are used for calculating y\

In [4]:
# PARAMETERS TO CHANGE
final_season = 2024 # the season we want to predict, so for out submission it will be 2025
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[i] for i in range(start_season, final_season+1)] # at which seasons to look at when calculating team's stats
days_back = 14 # how many days back from the start of tourney to calculate team's stats per season
maximum_favoured_seed = 1 # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14
include_men = True # include M... data sets when preparing x and y
include_women = False # include W... data sets when preparing x and y

In [5]:
# READ DATA
regular_results = pd.concat([
    MRegularSeasonDetailedResults.copy() if include_men else None,
    WRegularSeasonDetailedResults.copy() if include_women else None
], ignore_index=True)
tourney_results = pd.concat([
    MNCAATourneyDetailedResults.copy() if include_men else None,
    WNCAATourneyDetailedResults.copy() if include_women else None
], ignore_index=True)
seeds = pd.concat([
    MNCAATourneySeeds.copy() if include_men else None,
    WNCAATourneySeeds.copy() if include_women else None
], ignore_index=True)
number_of_added_columns = 0 # If you add any columns AFTER obtaining data from get_all_core_data change this counter

In [6]:
# GET ALL DATA NEEDED TO USE THE MODELS
df_train, df_test = get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
    start_season = start_season, season_years_list  = season_years_list, days_back = days_back,
    maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
    include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)

## How to add more columns to data frame

In [7]:
# ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN

if include_men:
    elo = pd.read_csv(join(data_path, 'elo.csv'))
    elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())

    def add_elo_column(df):
        df = df.copy()
        df = pd.merge(
                df,
                elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                left_on=['Season', 'DayNum', 'T1_TeamID'],
                right_on=['Season', 'DayNum', 'TeamID'],
                how='left'
            )
        df = df.drop(['TeamID'], axis=1)
        return df

    df_train = add_elo_column(df_train)
    df_test = add_elo_column(df_test)

    number_of_added_columns = 2

## How to get x and y

In [8]:
# Get x, y train and test fromd data frames
x_train, y_train = x_y_from_data_frame(df_train)
x_test, y_test = x_y_from_data_frame(df_test)

# Get data frames with columns present in x_train and x_test
df_train_x = df_train.iloc[:, 6:]
df_test_x = df_test.iloc[:, 6:]

# Clear NaN if it's present
x_train, y_train = clear_na_from_x_y(x_train.copy(), y_train.copy())
x_test, y_test = clear_na_from_x_y(x_test.copy(), y_test.copy())

In [9]:
df_train

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_opponent_Score,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_FTM,T1_opponent_FTA,T1_opponent_OR,T1_opponent_DR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_Blk,T1_opponent_PF,T1_PointDiff,T2_Score_mean,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_opponent_Score,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_FTM,T2_opponent_FTA,T2_opponent_OR,T2_opponent_DR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,2003,134,1421,92,1411,84,0,83.000000,28.000000,59.000000,7.500000,17.500000,19.500000,25.500000,13.000000,31.000000,16.000000,15.000000,7.000000,4.000000,18.500000,75.500000,26.000000,67.000000,9.500000,27.500000,14.000000,23.500000,16.500000,21.000000,15.500000,11.500000,8.000000,5.500000,19.500000,7.500000,75.200000,26.000000,51.000000,4.600000,12.600000,18.600000,31.200000,12.200000,26.400000,16.400000,16.400000,4.800000,4.000000,17.600000,68.000000,25.200000,64.400000,5.800000,20.200000,11.800000,18.200000,13.400000,20.000000,13.800000,12.800000,8.000000,2.20,22.800000,7.200000,1.000000,1.000000,16,16,0,1324.742776,1445.589445
1,2003,136,1112,80,1436,51,0,83.000000,30.333333,73.333333,6.000000,23.333333,16.333333,23.333333,16.000000,25.666667,18.333333,11.666667,11.000000,4.000000,18.666667,78.666667,26.666667,61.666667,6.333333,17.000000,19.000000,26.333333,11.666667,29.000000,13.333333,17.000000,5.666667,2.333333,20.666667,4.333333,68.000000,24.666667,49.666667,4.666667,13.000000,14.000000,21.000000,11.000000,25.333333,15.333333,13.000000,7.000000,2.000000,15.333333,56.000000,19.333333,52.333333,6.666667,20.333333,10.666667,16.333333,9.333333,17.333333,10.333333,11.333333,6.000000,3.00,17.000000,12.000000,0.666667,1.000000,1,16,-15,2087.437190,2125.806920
2,2003,136,1113,84,1272,71,0,82.333333,30.666667,62.333333,3.666667,11.666667,17.333333,24.000000,14.333333,24.333333,16.333333,14.666667,5.333333,3.666667,18.333333,74.666667,25.666667,56.000000,5.666667,15.333333,17.666667,22.333333,8.666667,18.666667,15.333333,15.666667,7.333333,2.666667,21.333333,7.666667,74.500000,24.500000,55.500000,8.500000,22.000000,17.000000,25.500000,12.500000,25.500000,16.000000,12.250000,4.750000,4.750000,22.250000,67.250000,23.500000,56.750000,4.250000,16.500000,16.000000,28.000000,14.250000,23.250000,10.750000,11.500000,8.000000,2.00,20.000000,7.250000,0.666667,0.750000,10,7,3,1810.546112,1843.235904
3,2003,136,1141,79,1166,73,0,83.400000,25.400000,49.200000,8.200000,17.800000,24.400000,30.600000,11.400000,25.000000,16.000000,16.800000,7.800000,4.400000,17.000000,66.400000,24.400000,60.600000,6.000000,15.800000,11.600000,16.000000,14.000000,16.200000,9.800000,15.600000,10.000000,2.200000,23.200000,17.000000,69.000000,25.333333,55.000000,5.000000,14.666667,13.333333,18.666667,13.333333,26.333333,14.666667,13.666667,7.000000,5.333333,16.666667,60.333333,23.666667,58.666667,4.333333,13.666667,8.666667,12.333333,11.666667,20.333333,12.000000,13.000000,6.333333,3.00,17.666667,8.666667,1.000000,1.000000,11,6,5,1704.724089,1729.865982
4,2003,136,1143,76,1301,74,0,63.666667,24.000000,60.333333,5.666667,17.000000,10.000000,15.333333,14.333333,27.000000,11.333333,14.666667,5.333333,3.666667,16.333333,65.666667,24.333333,66.000000,6.666667,22.666667,10.333333,15.666667,16.333333,24.333333,13.000000,10.666667,7.666667,3.333333,16.666667,-2.000000,74.000000,24.200000,50.400000,8.400000,19.400000,17.200000,22.000000,9.200000,23.800000,14.400000,14.400000,3.600000,4.000000,20.000000,74.000000,25.400000,57.600000,5.400000,15.600000,17.800000,23.200000,11.600000,20.200000,12.800000,8.800000,7.400000,2.80,21.000000,0.000000,0.333333,0.600000,8,9,-1,190

In [12]:
df_test

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_opponent_Score,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_FTM,T1_opponent_FTA,T1_opponent_OR,T1_opponent_DR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_Blk,T1_opponent_PF,T1_PointDiff,T2_Score_mean,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_opponent_Score,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_FTM,T2_opponent_FTA,T2_opponent_OR,T2_opponent_DR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,2024,134,1161,67,1438,42,0,75.000000,26.750000,55.000000,6.250000,21.000000,15.250000,21.750000,7.500000,21.250000,16.000000,9.250000,4.500000,3.750000,19.250000,71.750000,25.000000,54.500000,6.000000,17.750000,15.750000,20.250000,9.250000,21.000000,13.000000,8.500000,5.000000,3.750000,19.000000,3.250000,67.666667,26.333333,61.333333,9.666667,26.333333,5.333333,11.000000,7.333333,24.000000,18.333333,6.000000,6.666667,3.666667,11.0,63.333333,24.333333,58.666667,5.333333,22.333333,9.333333,10.666667,8.666667,25.666667,11.333333,8.666667,4.666667,4.666667,11.000000,4.333333,0.750000,0.666667,10,10,0,1766.448519,1865.132162
1,2024,134,1447,71,1224,68,0,60.000000,22.666667,53.333333,5.000000,17.666667,9.666667,15.666667,9.333333,25.666667,9.666667,9.000000,3.333333,2.666667,14.000000,53.333333,19.666667,55.000000,6.333333,26.000000,7.666667,12.000000,7.333333,20.666667,11.333333,8.333333,4.666667,3.666667,15.666667,6.666667,71.500000,21.000000,50.000000,8.000000,21.250000,21.500000,28.500000,8.750000,22.500000,12.250000,12.000000,4.500000,3.250000,18.5,70.750000,26.000000,56.250000,4.500000,12.750000,14.250000,21.750000,8.750000,17.000000,13.750000,9.500000,8.000000,4.500000,20.750000,0.750000,1.000000,0.750000,16,16,0,1310.062646,1450.749909
2,2024,135,1160,60,1129,53,0,70.000000,26.200000,56.200000,6.400000,17.200000,11.200000,15.400000,8.200000,25.600000,14.400000,9.400000,4.400000,1.200000,12.600000,63.400000,24.800000,58.600000,5.000000,20.200000,8.800000,13.400000,8.800000,20.000000,11.600000,7.600000,6.200000,4.200000,15.000000,6.600000,72.500000,24.000000,66.000000,10.500000,30.500000,14.000000,20.000000,14.000000,24.500000,13.000000,10.500000,5.000000,4.000000,19.0,76.500000,29.000000,65.500000,5.500000,22.500000,13.000000,20.500000,10.500000,28.000000,14.000000,11.000000,7.500000,5.500000,16.000000,-4.000000,0.800000,0.500000,10,10,0,1871.734585,1953.752712
3,2024,135,1212,88,1286,81,0,70.800000,22.000000,50.000000,6.600000,16.200000,20.200000,27.800000,6.400000,23.000000,10.600000,12.600000,7.200000,4.600000,16.200000,65.800000,22.600000,60.200000,6.000000,22.000000,14.600000,18.600000,9.800000,21.000000,10.800000,11.600000,8.000000,3.000000,20.200000,5.000000,83.333333,30.333333,59.000000,10.000000,25.666667,12.666667,16.000000,6.000000,24.666667,11.666667,8.666667,7.000000,3.666667,14.0,74.333333,27.333333,60.666667,9.000000,23.666667,10.666667,15.666667,9.333333,20.333333,10.333333,12.000000,4.666667,2.666667,15.333333,9.000000,0.800000,1.000000,16,16,0,1331.728720,1561.437890
4,2024,136,1112,85,1253,65,0,70.500000,24.000000,55.000000,6.750000,20.500000,15.750000,23.000000,8.500000,24.500000,14.750000,13.500000,6.750000,3.000000,16.250000,64.750000,24.000000,57.750000,6.250000,20.250000,10.500000,13.500000,6.000000,22.000000,11.250000,11.500000,9.250000,6.000000,19.500000,5.750000,79.000000,28.200000,63.200000,5.800000,18.000000,16.800000,22.600000,11.800000,23.600000,16.600000,9.200000,6.600000,4.200000,17.4,76.000000,26.200000,58.000000,7.600000,20.800000,16.000000,19.600000,6.400000,22.200000,14.400000,9.800000,5.200000,2.000000,19.800000,3.000000,0.500000,0.600000,2,15,-13,1993.065548,2049.325624
...,...,...,

In [10]:
x_train

array([[ 8.30000000e+01,  2.80000000e+01,  5.90000000e+01, ...,
         0.00000000e+00,  1.32474278e+03,  1.44558944e+03],
       [ 8.30000000e+01,  3.03333333e+01,  7.33333333e+01, ...,
        -1.50000000e+01,  2.08743719e+03,  2.12580692e+03],
       [ 8.23333333e+01,  3.06666667e+01,  6.23333333e+01, ...,
         3.00000000e+00,  1.81054611e+03,  1.84323590e+03],
       ...,
       [ 7.66666667e+01,  2.83333333e+01,  5.86666667e+01, ...,
         1.00000000e+00,  2.05025779e+03,  2.09170037e+03],
       [ 8.00000000e+01,  2.90000000e+01,  5.94000000e+01, ...,
         4.00000000e+00,  1.84020842e+03,  1.93928384e+03],
       [ 6.42500000e+01,  2.15000000e+01,  5.65000000e+01, ...,
         1.00000000e+00,  2.00366925e+03,  2.09632029e+03]])

In [13]:
x_test

array([[  75.        ,   26.75      ,   55.        , ...,    0.        ,
        1766.44851921, 1865.13216189],
       [  60.        ,   22.66666667,   53.33333333, ...,    0.        ,
        1310.06264618, 1450.74990884],
       [  70.        ,   26.2       ,   56.2       , ...,    0.        ,
        1871.73458525, 1953.75271228],
       ...,
       [  90.        ,   33.        ,   73.        , ...,    3.        ,
        2018.94042036, 2068.95188099],
       [  80.16666667,   27.5       ,   57.16666667, ...,   10.        ,
        1954.78353941, 1981.34106167],
       [  73.33333333,   23.66666667,   52.33333333, ...,    0.        ,
        2178.11347661, 2191.54686187]])

In [11]:
y_train

array([1., 1., 1., ..., 0., 0., 0.])

In [14]:
y_test

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## Important notes
We only look at detailed games for men and women.\
For women the records start from 2010.\
For men the records start from 2003.\
In 2020 there was no tournament. We shouldn't use data from this season.